In [10]:
import aiohttp
import asyncio
import nest_asyncio
import os
from dotenv import load_dotenv
from typing import List, Dict, Any, Tuple, Union
import json
from github import Github, Auth
from gidgethub.aiohttp import GitHubAPI
from gidgethub import GitHubException
import sys
import base64
from stamina import retry, retry_context
import re
from functools import partial
from textacy import preprocessing
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from qdrant_client.http.models import Distance, VectorParams
import uuid
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langgraph.graph import START, StateGraph
from langchain_core.documents import Document
from typing_extensions import List, TypedDict

import numpy as np
import matplotlib.pyplot as plt

# allow for multiple event loops within Jupyter
nest_asyncio.apply()

# load environment variables
load_dotenv()

import pickle

In [14]:
def _is_retriable(exc: Exception) -> bool:
    """Retry on transient network/server errors; skip deterministic 404/403."""
    if isinstance(exc, GitHubException):
        return exc.status_code not in (404, 403)
    return isinstance(exc, aiohttp.ClientError)


@retry(on=_is_retriable, attempts=3, wait_initial=0.5, wait_max=10.0)
async def get_root_markdown_files(gh: GitHubAPI, owner: str, repo: str) -> List[Dict[Any, Any]]:
    """
    Get all markdown files from the root directory of a repository.
    This includes README.md, CONTRIBUTING.md, CHANGELOG.md, etc.
    """
    try:
        # Get contents of root directory
        contents = await gh.getitem(f"/repos/{owner}/{repo}/contents/")

        # Filter for markdown files (case-insensitive)
        markdown_files = [
            file for file in contents
            if file['type'] == 'file' and file['name'].lower().endswith('.md')
        ]

        return markdown_files
    except GitHubException as e:
        print(f"Error fetching contents for {owner}/{repo}: {e}")
        return []


@retry(on=_is_retriable, attempts=3, wait_initial=0.5, wait_max=10.0)
async def fetch_markdown_content(gh: GitHubAPI, owner: str, repo: str, file_path: str):
    """Fetch content of a specific markdown file"""
    try:
        file_data = await gh.getitem(f"/repos/{owner}/{repo}/contents/{file_path}")
        content = base64.b64decode(file_data['content']).decode('utf-8')

        return {
            'name': file_data['name'],
            'path': file_data['path'],
            'size': file_data['size'],
            'content': content,
            'success': True
        }
    except GitHubException as e:
        return {
            'path': file_path,
            'success': False,
            'error': str(e)
        }


async def fetch_starred_repos_with_docs(max_repos: int = None, concurrent_tasks: int = 20) -> List[Dict[Any, Any]]:
    """
    Fetch all starred repositories for the authenticated GitHub user, then
    concurrently retrieve documentation for each repo using the following strategy:

      1. Try the /readme endpoint first (covers README.md, README.rst, etc.)
      2. If no README exists (404), fall back to fetching ALL markdown files
         found in the root directory of the repository.
      3. If neither exists, the repo is recorded with an empty docs list.

    All per-repo fetches run concurrently via asyncio.gather, so even a large
    starred list is handled as fast as the GitHub rate limit allows.

    Args:
        max_repos: Optional cap on the number of starred repos to process.
                   Defaults to None (fetch all starred repos).

    Returns:
        A list of dicts, one per repo, with keys:
            repo         - "owner/name"
            description  - repo description string
            stars        - stargazer count
            language     - primary language
            url          - HTML URL
            doc_source   - "readme" | "root_markdown" | None
            docs         - list of {name, path, size, content} dicts
    """
    oauth_token = os.getenv("GITHUB_TOKEN")

    # Semaphore pattern for throttling concurrent asyncio tasks
    semaphore = asyncio.Semaphore(concurrent_tasks)

    async with aiohttp.ClientSession() as session:
        gh = GitHubAPI(session, "markdown-fetcher", oauth_token=oauth_token)

        # ── Step 1: Collect starred repos (sequential; getiter handles pagination) ──
        print("Fetching starred repositories...")
        starred_repos: List[Dict[Any, Any]] = []
        async for repo in gh.getiter("/user/starred", accept="application/vnd.github.mercy-preview+json"):
            starred_repos.append(repo)
            if max_repos and len(starred_repos) >= max_repos:
                break

        print(f"Found {len(starred_repos)} starred repositories")
        print("Fetching documentation for all repos concurrently...\n")

        # ── Step 2: Define per-repo coroutine ──────────────────────────────────────
        async def fetch_repo_docs(repo: Dict[Any, Any]) -> Dict[Any, Any]:
            owner = repo["owner"]["login"]
            name = repo["name"]
            full_name = repo["full_name"]
            topics: List[str] = repo["topics"]

            base = {
                "repo": full_name,
                "description": repo.get("description"),
                "topics": topics,
                "stars": repo.get("stargazers_count"),
                "language": repo.get("language"),
                "url": repo.get("html_url"),
            }

            # Try README first via the dedicated /readme endpoint
            try:
                async for attempt in retry_context(
                    on=_is_retriable, attempts=3, wait_initial=0.5, wait_max=10.0
                ):
                    with attempt:
                        readme_data = await gh.getitem(f"/repos/{owner}/{name}/readme")
                content = base64.b64decode(readme_data["content"]).decode("utf-8")
                return {
                    **base,
                    "doc_source": "readme",
                    "docs": [
                        {
                            "name": readme_data["name"],
                            "path": readme_data["path"],
                            "size": readme_data["size"],
                            "content": content,
                        }
                    ],
                }
            except GitHubException as e:
                if e.status_code != 404:
                    # Surface unexpected errors without crashing the whole gather
                    print(f"  Warning: unexpected error fetching README for {full_name}: {e}")

            # README absent — fall back to all root-level .md files
            markdown_files = await get_root_markdown_files(gh, owner, name)
            if markdown_files:
                tasks = [
                    fetch_markdown_content(gh, owner, name, f["path"])
                    for f in markdown_files
                ]
                file_results = await asyncio.gather(*tasks)
                return {
                    **base,
                    "doc_source": "root_markdown",
                    "docs": [r for r in file_results if r.get("success")],
                }

            # No documentation found at all
            return {**base, "doc_source": None, "docs": []}

        # ── Step 3: Fan out — all repos fetched concurrently ──────────────────────
        async def fetch_repo_docs_throttled(repo: Dict[Any, Any]) -> Dict[Any, Any]:
            async with semaphore:
                return await fetch_repo_docs(repo)

        results: List[Dict[Any, Any]] = await asyncio.gather(
            *[fetch_repo_docs_throttled(repo) for repo in starred_repos]
        )

        # ── Summary ────────────────────────────────────────────────────────────────
        with_readme = [r for r in results if r["doc_source"] == "readme"]
        with_md = [r for r in results if r["doc_source"] == "root_markdown"]
        no_docs = [r for r in results if r["doc_source"] is None]

        print(f"\n{'='*70}")
        print("DOCUMENTATION FETCH SUMMARY")
        print(f"{'='*70}")
        print(f"Total repos processed : {len(results)}")
        print(f"  README found        : {len(with_readme)}")
        print(f"  Root markdown files : {len(with_md)}")
        print(f"  No docs found       : {len(no_docs)}")
        if gh.rate_limit:
            print(f"Rate limit remaining  : {gh.rate_limit.remaining}")

        return results


def strip_markdown(text: str) -> str:
    """Remove common Markdown syntax; keep inner text and emojis."""
    # Headings: ## Title -> Title
    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.MULTILINE)
    # Inline code: `code` -> code
    text = re.sub(r"`([^`]+)`", r"\1", text)
    # Bold/italic: **bold** or *italic* -> bold / italic
    text = re.sub(r"\*{1,2}([^*]+)\*{1,2}", r"\1", text)
    text = re.sub(r"_{1,2}([^_]+)_{1,2}", r"\1", text)
    # Link markup: [text](url) -> text
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    # Collapse many newlines
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text


def make_normalize_text_pipeline(
    *,
    url_repl: str = "",
    unicode_form: str = "NFC",
):
    """
    Build a textacy pipeline: Markdown/HTML → plain text (emojis kept).

    Args:
        url_repl: Replacement for URLs (default ""). Use "_URL_" to keep a placeholder.
        unicode_form: Unicode normalization form ("NFC", "NFD", "NFKC", "NFKD"). Default "NFC".

    Returns:
        A callable that takes a string and returns preprocessed plain text.
    """
    return preprocessing.make_pipeline(
        strip_markdown,
        preprocessing.remove.html_tags,
        preprocessing.normalize.bullet_points,
        preprocessing.normalize.quotation_marks,
        partial(preprocessing.normalize.unicode, form=unicode_form),
        preprocessing.normalize.whitespace,
    )


# Default pipeline instance
normalize_text_pipeline = make_normalize_text_pipeline()

In [6]:
MAX_REPOS = 3000

starred_repo_data = asyncio.run(fetch_starred_repos_with_docs(MAX_REPOS))

Fetching starred repositories...
Found 2049 starred repositories
Fetching documentation for all repos concurrently...


DOCUMENTATION FETCH SUMMARY
Total repos processed : 2049
  README found        : 2041
  Root markdown files : 0
  No docs found       : 8
Rate limit remaining  : 2874


In [7]:
# save the loaded github data from my account to pickled file
import pickle

with open("github_data.pkl", "wb") as f:
    pickle.dump(starred_repo_data, f)

In [8]:
# define embedding model for vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
MAX_CHARACTERS = 30_000  # fall-back for when text is too long

In [24]:
NAMESPACE = uuid.NAMESPACE_URL

def repo_to_uuid(repo_name: str) -> str:
    return str(uuid.uuid5(NAMESPACE, repo_name))

def normalize_docs(docs: List[Dict[str, Any]]):
    content_str = ''
    for doc in docs:
        content_str += (doc['content'] + '\n\n')
    truncated = content_str[:MAX_CHARACTERS] if len(content_str) > MAX_CHARACTERS else content_str
    normalized_text = normalize_text_pipeline(truncated)
    return normalized_text


def organize_repo_data(repo_data: List[Dict[Any, Any]]) -> Tuple[List[str], List[Dict[str, Union[str, int]]], List[str]]:
    ids = []
    metadata = []
    docs = []
    for repo in repo_data:
        id = repo_to_uuid(repo['repo'])
        description = repo['description']
        topics = repo['topics']
        doc_source = repo['doc_source']
        stars = repo['stars']
        url = repo['url']
        language = repo['language']
        # append to lists for DB consumption
        ids.append(id)
        metadata.append({
            'id': id,
            'repo': repo['repo'],
            'description': description,
            'topics': topics,
            'language': language,
            'doc_source': doc_source,
            'stars': stars,
            'url': url,
        })
        topics_str = f"Topics: {','.join(topics)}\n"
        docs.append(topics_str + normalize_docs(repo['docs']))
    return ids, metadata, docs

In [25]:
ids, metadata, docs = organize_repo_data(starred_repo_data)

In [26]:
print(f"id: {ids[0]}\n\nmetadata: {metadata[0]}\n\ndocs:\n {docs[0]}")

id: a5ef9890-4c52-5a83-8aa1-c1b266a4f6fb

metadata: {'id': 'a5ef9890-4c52-5a83-8aa1-c1b266a4f6fb', 'repo': 'pymc-devs/pytensor', 'description': 'PyTensor allows you to define, optimize, and efficiently evaluate mathematical expressions involving multi-dimensional arrays.', 'topics': ['ai', 'bayesian-inference', 'computational-science', 'deep-learning', 'statistics'], 'language': 'Python', 'doc_source': 'readme', 'stars': 594, 'url': 'https://github.com/pymc-devs/pytensor'}

docs:
 Topics: ai,bayesian-inference,computational-science,deep-learning,statistics
.. image:: https://cdn.rawgit.com/pymc-devs/pytensor/main/doc/images/PyTensorRGB.svg
 :height: 100px
 :alt: PyTensor logo
 :align: center
|Tests Status| |Coverage|
|Project Name| is a Python library that allows one to define, optimize, and
efficiently evaluate mathematical expressions involving multi-dimensional arrays.
It provides the computational backend for PyMC .
Features
- A hackable, pure-Python codebase
- Extensible graph fra

In [27]:
# build in-memory vector store # and load documents (repo content and metadata associated)
client = QdrantClient(":memory:")
client.create_collection(
    collection_name="ask_my_bookmark",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="ask_my_bookmark",
    embedding=embeddings,
)

_ = vector_store.add_texts(texts=docs, ids=ids, metadatas=metadata)
# define retriever
naive_retriever = vector_store.as_retriever(search_kwargs={"k" : 10})

In [28]:
### build RAG graph
def format_context(docs) -> str:
    chunks = []
    for doc in docs:
        meta = doc.metadata
        topics = meta.get('topics', [])
        if topics:
            topics_str = f"Topics: {','.join(topics)}"
        else:
            topics_str = "N/A"
        chunk = f"""---
Repo: {meta.get('repo', 'N/A')}
URL: {meta.get('url', 'N/A')}
Description: {meta.get('description', 'N/A')}
Topics: {topics_str}
Programming Language: {meta.get('language', 'N/A')}
Stars: {meta.get('stars', 'N/A')}

README excerpt:
{doc.page_content.strip()}
---"""
        chunks.append(chunk)
    return "\n\n".join(chunks)


# define the RAG Pipeline in LangChain
RAG_SYSTEM_PROMPT = """
You are AskMyBookmark, a personal research assistant with access to the user's GitHub starred repositories.

Your job is to help the user discover, recall, and explore repositories they have bookmarked on GitHub. You answer questions by reasoning over the retrieved repository context provided to you — not from your general knowledge of what exists on GitHub.

**Ground rules:**
- Only surface repositories that appear in the retrieved context below. Do not invent or suggest repositories that are not present in the context.
- If no retrieved repositories are relevant to the query, say so honestly and suggest the user try rephrasing or broadening their search.
- You may use your general knowledge to explain a topic or technology, but all repository recommendations must come exclusively from the retrieved context.

**When presenting results:**
- Always include the repository's full name (Repo) as a markdown link to its GitHub URL: [Repo](URL)
- Include a brief description of what the repo does (from the description and topics fields), written in your own words if the original description is terse or absent.
- Explain in 1–2 sentences *why* this repository is relevant to the user's query — this is the most important part.
- Group or rank results by relevance if there are several.
- If useful, note the primary programming language, star count, or topics to help the user evaluate the match.

**Tone:** Conversational, concise, and helpful. Treat the user as a developer who starred these repos intentionally and wants quick, intelligent recall — not a tutorial.

---

Retrieved repository context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(RAG_SYSTEM_PROMPT),
    HumanMessagePromptTemplate.from_template("{question}"),
])
llm = ChatOpenAI(model="gpt-4.1-nano")

def generate(state):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
    response = llm.invoke(messages)
    return {"response" : response.content}

class State(TypedDict):
    question: str
    context: List[Document]
    response: str

def retrieve(state):
    retrieved_docs = naive_retriever.invoke(state["question"])
    return {"context" : retrieved_docs}

# build graph
rag_graph_builder = StateGraph(State).add_sequence([retrieve, generate])
rag_graph_builder.add_edge(START, "retrieve")
rag_graph = rag_graph_builder.compile()

In [29]:
response = rag_graph.invoke({"question" : "Do I have any starred repositories related to Bayesian Statistics or Bayesian Modeling?"})
print(response['response'])

Yes, you have starred repositories related to Bayesian Statistics and Bayesian Modeling. Here are some of the relevant ones:

- [PyMC Resources](https://github.com/pyMC-devs/resources): This repository offers educational resources for PyMC, including ports of several Bayesian modeling books like "Bayesian Modeling and Computation in Python" and "Bayesian Statistical Methods." It is highly relevant if you are interested in Bayesian modeling in Python.
- [bayes-toolbox](https://github.com/kim-hyosub/bayes-toolbox): A Python package for running sophisticated Bayesian analyses in a straightforward way, developed by Hyosub E. Kim. It is tailored for Bayesian statistical workflows.
- [Bayesian Data Analysis course material](https://github.com/avehtari/BDAcourse): This repository contains course materials for Bayesian Data Analysis, based on the book by Gelman et al., with code examples in Python.
- [Bayesian Analysis with Python - Second Edition](https://github.com/astokk/Bayesian-Analysis-w

In [30]:
response2 = rag_graph.invoke({"question" : "What are some top deep learning libraries that I have starred on github?"})
print(response2['response'])

Based on your starred repositories, some top deep learning libraries you have bookmarked include:

- [Pixellib](https://github.com/ayoolaolaperson/PixelLib) — A library for performing segmentation of objects in images and videos, supporting both Mask R-CNN and PointRend architectures. It's relevant for deep learning applications in computer vision, especially instance and semantic segmentation.
- [Caffe](https://github.com/BVLC/caffe) — A deep learning framework designed for speed and modularity, particularly popular in computer vision tasks.
- [Deep Neural Networks library (Sonnet)](https://github.com/deepmind/sonnet) — TensorFlow-based neural network library by DeepMind for building complex models.
- [Flax](https://github.comGoogle/flax) — A neural network library built on JAX, optimized for flexibility and research.
- [TorchVision](https://github.com/pytorch/vision) — (implied in your list, often starred in conjunction with PyTorch) — a PyTorch package with models, datasets, and ima

In [31]:
response3 = rag_graph.invoke({"question" : "Can you tell me how many times the PyTorch Library has been starred on Github versus the Tensorflow library?"})
print(response3['response'])

Based on your starred repositories, the PyTorch library has been starred approximately 94K times on GitHub. The TensorFlow library, on the other hand, has about 200K stars. So, TensorFlow has significantly more stars than PyTorch in your starred repositories.


In [32]:
response4 = rag_graph.invoke({"question" : "What are the top ""awesome"" github repositories that I have starred?"})
print(response4['response'])

Based on the repositories you've starred, here are some of the top ones that stand out:

1. [Awesome-Kubernetes](https://github.com/sindresorhus/awesome)  
A curated list of Kubernetes resources, including guides, tools, and best practices. It's relevant if you're interested in container orchestration and cloud-native development.

2. [30 seconds of code](https://github.com/30secondsofcode/awesome)  
A collection of concise coding articles and snippets across multiple languages and topics, great for quick learning and reference.

3. [Flyte](https://github.com/flyteorg/flyte)  
An open-source cloud-native platform for building scalable data and machine learning pipelines on Kubernetes, perfect if you're into workflows and MLops.

4. [Awesome Hadoop !](https://github.com/sindresorhus/awesome)  
A comprehensive list of Hadoop and big data resources, useful if you're working with large-scale data processing.

5. [PyGitHub](https://github.com/PyGithub/PyGithub)  
A Python library to manage 

In [33]:
response5 = rag_graph.invoke({"question" : "What are the top machine learning repositories I have starred?"})
print(response5['response'])

Based on your starred repositories, here are some of the top machine learning-related repositories:

1. [Google Research](https://github.com/google-research/google-research) — Contains code from Google Research on various AI topics. It's a large collection of cutting-edge projects related to machine learning and deep learning, widely used for research and development.

2. [Complete Machine Learning Package](https://github.com/ageron/handson-ml) — A highly ranked, comprehensive repository with tutorials, notebooks, and tools for classical and deep machine learning, aimed at practical implementation.

3. [Libra](https://github.com/awslabs/libra) — A Python library that automates machine learning processes with one-liners, useful for quick model creation and evaluation.

4. [500+ Artificial Intelligence Project List](https://github.com/ashishpatel26/500-AI-Machine-learning-Deep-learning-Computer-vision-NLP-Projects-with-code) — A curated list of AI/ML projects across various domains inclu

In [34]:
response6 = rag_graph.invoke({"question" : "What are the top machine learning Python libraries I have starred? Please do not include repositories that are just educational resources or educational materials?"})
print(response6['response'])

Based on your starred repositories, the top machine learning Python libraries you have starred are:

- [NumPy](https://github.com/numpy/numpy) — The fundamental package for scientific computing with Python, widely used for array and matrix operations. Your star indicates you value its essential role in data manipulation and numerical computation in ML workflows.
- [scikit-learn](https://github.com/scikit-learn/scikit-learn) — A popular library for classical machine learning algorithms, providing simple and efficient tools for data mining and analysis. Your star reflects its importance in your ML projects.
- [TensorFlow](https://github.com/tensorflow/tensorflow) — An open-source deep learning framework used for building and deploying neural networks. Your starred TensorFlow repositories showcase your interest in deep learning and neural networks.
- [Keras](https://github.com/keras-team/keras) — A high-level neural networks API that runs on top of TensorFlow, making model design easier. 

In [35]:
response7 = rag_graph.invoke({"question" : "Do I have any R programming language libraries or packages starred on github?"})
print(response7['response'])

Yes, you have starred several R packages related to data analysis and manipulation. Notably:

- [tidypolars for R](https://github.com/robertjgraham/tidypolars): This package provides an interface to Polars, a fast DataFrame library, with a syntax familiar to R tidyverse users. It’s relevant if you're exploring fast DataFrame operations in R.

- [rpolars for R](https://github.com/robertjgraham/rpolars): This package allows using Polars DataFrames directly from R, offering a high-performance alternative to base R or data.table.

- [polarssql](https://github.com/robertjgraham/polarssql): An experimental package that provides a DBI interface to Polars, enabling SQL-based workflows in R with Polars.

- [r-polars-dashboard](https://github.com/robertjgraham/r-polars-dashboard): A dashboard tool comparing R and Python Polars APIs, useful for understanding cross-language features.

- [neo-r-polars](https://github.com/robertjgraham/neo-r-polars): The next-generation R API for Polars, focusing on

In [36]:
response8 = rag_graph.invoke({"question" : "Are there any R programming language libraries starred that are not related to data analysis?"})
print(response8['response'])

Based on the retrieved context, the R programming language repositories with stars are mostly related to data analysis and visualization. Specifically, the cheat sheets and data packages listed for R (like tidiverse, data.table, lubridate, and data import/transformation tools) focus on data analysis, cleaning, and visualization tasks.

There are no starred R repositories mentioned that are explicitly unrelated to data analysis, so it seems the starred R projects here are primarily centered on data analysis, visualization, and related tasks. Would you like to explore specific types of R libraries or datasets that might be outside data analysis?


#### Let's try an advanced retrieval strategy
- hybrid search with an ensemble retriever
  - 1 dense retrieval utilizing cosine-similarity embeddings (k=10)
  - 1 BM25 (k=10) retriever (will increase matches on rare keywords that may be missed with only dense vector retrieval methods)
  - and reranking with Cohere's reranking ML model (k=10 best candidates)
  - no chunking strategy for now
    - next step could be trying a chunking strategy for markdown documents based on document length (but probably unnecessary for most repos)

In [15]:
import aiohttp
import asyncio
import nest_asyncio
import os
from dotenv import load_dotenv
from typing import List, Dict, Any, Tuple, Union
import json
from github import Github, Auth
from gidgethub.aiohttp import GitHubAPI
from gidgethub import GitHubException
import sys
import base64
from stamina import retry, retry_context
import re
from functools import partial
from textacy import preprocessing
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from qdrant_client.http.models import Distance, VectorParams
import uuid
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langgraph.graph import START, StateGraph
from langchain_core.documents import Document
from typing_extensions import List, TypedDict
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_cohere import CohereRerank
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever

import numpy as np
import matplotlib.pyplot as plt

# allow for multiple event loops within Jupyter
nest_asyncio.apply()

# load environment variables
load_dotenv()

import pickle

In [11]:
MAX_CHARACTERS = 30_000
NAMESPACE = uuid.NAMESPACE_URL

def strip_markdown(text: str) -> str:
    """Remove common Markdown syntax; keep inner text and emojis."""
    # Headings: ## Title -> Title
    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.MULTILINE)
    # Inline code: `code` -> code
    text = re.sub(r"`([^`]+)`", r"\1", text)
    # Bold/italic: **bold** or *italic* -> bold / italic
    text = re.sub(r"\*{1,2}([^*]+)\*{1,2}", r"\1", text)
    text = re.sub(r"_{1,2}([^_]+)_{1,2}", r"\1", text)
    # Link markup: [text](url) -> text
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    # Collapse many newlines
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text


def make_normalize_text_pipeline(
    *,
    url_repl: str = "",
    unicode_form: str = "NFC",
):
    """
    Build a textacy pipeline: Markdown/HTML → plain text (emojis kept).

    Args:
        url_repl: Replacement for URLs (default ""). Use "_URL_" to keep a placeholder.
        unicode_form: Unicode normalization form ("NFC", "NFD", "NFKC", "NFKD"). Default "NFC".

    Returns:
        A callable that takes a string and returns preprocessed plain text.
    """
    return preprocessing.make_pipeline(
        strip_markdown,
        preprocessing.remove.html_tags,
        preprocessing.normalize.bullet_points,
        preprocessing.normalize.quotation_marks,
        partial(preprocessing.normalize.unicode, form=unicode_form),
        preprocessing.normalize.whitespace,
    )


# Default pipeline instance
normalize_text_pipeline = make_normalize_text_pipeline()

def repo_to_uuid(repo_name: str) -> str:
    return str(uuid.uuid5(NAMESPACE, repo_name))

def normalize_docs(docs: List[Dict[str, Any]]):
    content_str = ''
    for doc in docs:
        content_str += (doc['content'] + '\n\n')
    truncated = content_str[:MAX_CHARACTERS] if len(content_str) > MAX_CHARACTERS else content_str
    normalized_text = normalize_text_pipeline(truncated)
    return normalized_text


def organize_repo_data(repo_data: List[Dict[Any, Any]]) -> Tuple[List[str], List[Dict[str, Union[str, int]]], List[str]]:
    ids = []
    metadata = []
    docs = []
    for repo in repo_data:
        id = repo_to_uuid(repo['repo'])
        description = repo['description']
        topics = repo['topics']
        doc_source = repo['doc_source']
        stars = repo['stars']
        url = repo['url']
        language = repo['language']
        # append to lists for DB consumption
        ids.append(id)
        metadata.append({
            'id': id,
            'repo': repo['repo'],
            'description': description,
            'topics': topics,
            'language': language,
            'doc_source': doc_source,
            'stars': stars,
            'url': url,
        })
        topics_str = f"Topics: {','.join(topics)}\n"
        docs.append(topics_str + normalize_docs(repo['docs']))
    return ids, metadata, docs

In [12]:
# load cached data
with open("github_data.pkl", "rb") as f:
    starred_repo_data = pickle.load(f)

In [13]:
ids, metadata, docs = organize_repo_data(starred_repo_data)

In [14]:
# generate langchain documents from texts, ids and metadata
documents = [
    Document(page_content=text, metadata={**meta, "id": id_})
    for text, id_, meta in zip(docs, ids, metadata)
]

In [16]:
# build BM25 retriever
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 10

In [17]:
# build naive dense vector retriever
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
client = QdrantClient(":memory:")
client.create_collection(
    collection_name="ask_my_bookmark",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="ask_my_bookmark",
    embedding=embeddings,
)
vector_store.add_documents(documents)
dense_retriever = vector_store.as_retriever(search_kwargs={"k": 10})

In [18]:
# build ensemble retriever which utilizes Reciprocal Rank Fusion under the hood to combine results from both retrievers
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5,0.5]
)

In [37]:
### build RAG graph
def format_context(docs) -> str:
    chunks = []
    for doc in docs:
        meta = doc.metadata
        topics = meta.get('topics', [])
        if topics:
            topics_str = f"Topics: {','.join(topics)}"
        else:
            topics_str = "N/A"
        chunk = f"""---
Repo: {meta.get('repo', 'N/A')}
URL: {meta.get('url', 'N/A')}
Description: {meta.get('description', 'N/A')}
Topics: {topics_str}
Programming Language: {meta.get('language', 'N/A')}
Stars: {meta.get('stars', 'N/A')}

README excerpt:
{doc.page_content.strip()}
---"""
        chunks.append(chunk)
    return "\n\n".join(chunks)


# define the RAG Pipeline in LangChain
RAG_SYSTEM_PROMPT = """
You are AskMyBookmark, a personal research assistant with access to the user's GitHub starred repositories.

Your job is to help the user discover, recall, and explore repositories they have bookmarked on GitHub. You answer questions by reasoning over the retrieved repository context provided to you — not from your general knowledge of what exists on GitHub.

**Ground rules:**
- Only surface repositories that are directly starred by the user. A directly starred repository is identified by the `repo` field in the retrieved context (e.g., `owner/repo-name`).
- Do NOT treat tools, libraries, papers, or projects that are merely *mentioned or listed inside* a repository's README as starred repositories. If a README is a curated list, awesome list, or ranked collection (e.g., its description contains phrases like "ranked list", "curated list", "awesome", "collection of links", or "resources"), treat that entire repository as a single starred item — do not extract or present its listed contents as individual starred repos.
- If no retrieved repositories are directly relevant to the query, say so honestly and suggest the user try rephrasing or broadening their search.
- You may use your general knowledge to explain a topic or technology, but all repository recommendations must come exclusively from the directly starred repos in the retrieved context.

**When presenting results:**
- Always include the repository's full name (Repo) as a markdown link to its GitHub URL: [Repo](URL)
- Include a brief description of what the repo does (from the description and topics fields), written in your own words if the original description is terse or absent.
- If the repository is a curated/awesome list, clearly label it as such (e.g., "curated list") so the user understands it is a collection, not a standalone library.
- Explain in 1–2 sentences *why* this repository is relevant to the user's query — this is the most important part.
- Group or rank results by relevance if there are several.
- If useful, note the primary programming language, star count, or topics to help the user evaluate the match.

**Tone:** Conversational, concise, and helpful. Treat the user as a developer who starred these repos intentionally and wants quick, intelligent recall — not a tutorial.

---

Retrieved repository context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(RAG_SYSTEM_PROMPT),
    HumanMessagePromptTemplate.from_template("{question}"),
])
llm = ChatOpenAI(model="gpt-4.1-nano")

def generate(state):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
    response = llm.invoke(messages)
    return {"response" : response.content}

class State(TypedDict):
    question: str
    context: List[Document]
    response: str

def retrieve(state):
    compressor = CohereRerank(model="rerank-v3.5")
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=ensemble_retriever, search_kwargs={"k": 10}
    )
    retrieved_docs = compression_retriever.invoke(state["question"])
    return {"context" : retrieved_docs}

# build graph
rag_graph_builder = StateGraph(State).add_sequence([retrieve, generate])
rag_graph_builder.add_edge(START, "retrieve")
rag_graph = rag_graph_builder.compile()

In [38]:
# evaluate responses
response = rag_graph.invoke({"question" : "Do I have any starred repositories related to Bayesian Statistics or Bayesian Modeling?"})
print(response['response'])

Yes, you have starred repositories related to Bayesian statistics and modeling. Specifically, you have starred the [PyMC](https://github.com/pymc-devs/pymc) repository. 

**[PyMC](https://github.com/pymc-devs/pymc)** is a highly relevant and comprehensive Python package for Bayesian statistical modeling. It focuses on advanced Markov chain Monte Carlo (MCMC) and variational inference algorithms, making it a powerful tool for Bayesian data analysis. This repository is a curated list and resource for Bayesian modeling, and its topics include Bayesian inference, probabilistic programming, and statistical analysis.

This starred repo aligns perfectly with your interest in Bayesian methods and is a trusted resource used widely in the Bayesian community.


In [39]:
for doc in response['context']:
    print(doc.metadata)
    print("\n\n")

{'id': 'c0875bd4-3277-561d-a9bd-3f9ba3a84065', 'repo': 'pymc-devs/pymc', 'description': 'Bayesian Modeling and Probabilistic Programming in Python', 'topics': ['bayesian-inference', 'mcmc', 'probabilistic-programming', 'pytensor', 'python', 'statistical-analysis', 'variational-inference'], 'language': 'Python', 'doc_source': 'readme', 'stars': 9504, 'url': 'https://github.com/pymc-devs/pymc', 'relevance_score': 0.62906903}



{'id': 'e901860a-7ff8-5671-84cc-2b760d76680b', 'repo': 'pymc-devs/pymc-resources', 'description': 'PyMC educational resources', 'topics': ['bayesian-inference', 'bayesian-statistics', 'data-analysis', 'data-science'], 'language': 'Jupyter Notebook', 'doc_source': 'readme', 'stars': 2076, 'url': 'https://github.com/pymc-devs/pymc-resources', '_id': '538b4c1f02b341ebbafdab651ba5b724', '_collection_name': 'ask_my_bookmark', 'relevance_score': 0.49245724}



{'id': 'ec85bbfa-0db6-550c-852a-1b5a4d6c6b9c', 'repo': 'datasciencemasters/go', 'description': 'The Open Source

In [40]:
response2 = rag_graph.invoke({"question" : "What are some repos related to recommender systems that I have starred?"})
print(response2['response'])

Here are some repositories related to recommender systems that you've starred:

1. [ListofRecommenderSystems](https://github.com/grahamjenson/listofrecommendersystems)  
   - A comprehensive list of various recommender system projects, benchmarks, and applications curated for easy exploration.  
   - This repo is useful for discovering different approaches and seeing what others have built or used in the field.

2. [Deep-Learning-for-Recommendation-Systems](https://github.com/robi56/Deep-Learning-for-Recommendation-Systems)  
   - A collection of papers, tutorials, and software focused on deep learning techniques applied to recommender systems.  
   - It’s helpful if you're interested in cutting-edge neural network approaches in recommendation.

3. [RecommenderSystem-Paper](https://github.com/daicoolb/RecommenderSystem-Paper)  
   - A repository for papers, tools, and frameworks related to recommender systems.  
   - Great for research or implementing new algorithms inspired by academi

In [41]:
for doc in response2['context']:
    print(doc.metadata)
    print("\n\n")

{'id': 'bc96cfad-0378-51f9-824a-1caadcd9c0f2', 'repo': 'jihoo-kim/awesome-RecSys', 'description': 'A curated list of awesome Recommender System (Books, Conferences, Researchers, Papers, Github Repositories, Useful Sites, Youtube Videos)', 'topics': [], 'language': None, 'doc_source': 'readme', 'stars': 1445, 'url': 'https://github.com/jihoo-kim/awesome-RecSys', '_id': 'f8b6ba6c3d2e4f95a5a433e795c346e5', '_collection_name': 'ask_my_bookmark', 'relevance_score': 0.8302782}



{'id': 'e3877564-0dcb-5cc3-be05-6ed27c5556c1', 'repo': 'recommenders-team/recommenders', 'description': 'Best Practices on Recommendation Systems', 'topics': ['ai', 'artificial-intelligence', 'data-science', 'deep-learning', 'jupyter-notebook', 'kubernetes', 'machine-learning', 'operationalization', 'python', 'ranking', 'rating', 'recommendation', 'recommendation-algorithm', 'recommendation-engine', 'recommendation-system', 'recommender', 'tutorial'], 'language': 'Python', 'doc_source': 'readme', 'stars': 21485, 'ur

In [42]:
response3 = rag_graph.invoke({"question" : "What are the top recommender systems libraries that I have starred in the past?"})
print(response3['response'])

Based on your starred repositories, here are some of the top recommender systems libraries you have starred:

1. [Surprise](https://github.com/NicolasHug/Surprise) — A popular Python library for building and analyzing collaborative filtering recommender systems. It includes many algorithms and is widely used for research and prototyping.

2. [LightFM](https://github.com/lyst/lightfm) — An active Python library implementing factorization and hybrid recommendation algorithms, suitable for both implicit and explicit feedback, known for its scalability and ease of use.

3. [Recommenders](https://github.com/recommenders-team/recommenders) — A comprehensive project with example notebooks, best practices, and implementations of various recommendation algorithms, aimed at researchers and practitioners for prototyping and production.

4. [Spotlight](https://github.com/maciejkula/spotlight) — A Python framework utilizing neural network-based models for recommendation tasks, supporting matrix fac

In [43]:
for doc in response3['context']:
    print(doc.metadata)
    print("\n\n")

{'id': 'bc96cfad-0378-51f9-824a-1caadcd9c0f2', 'repo': 'jihoo-kim/awesome-RecSys', 'description': 'A curated list of awesome Recommender System (Books, Conferences, Researchers, Papers, Github Repositories, Useful Sites, Youtube Videos)', 'topics': [], 'language': None, 'doc_source': 'readme', 'stars': 1445, 'url': 'https://github.com/jihoo-kim/awesome-RecSys', '_id': 'f8b6ba6c3d2e4f95a5a433e795c346e5', '_collection_name': 'ask_my_bookmark', 'relevance_score': 0.798077}



{'id': 'a77fae33-5ac7-5d81-81ce-597eec09e2d8', 'repo': 'grahamjenson/list_of_recommender_systems', 'description': 'A List of Recommender Systems and Resources', 'topics': [], 'language': None, 'doc_source': 'readme', 'stars': 4815, 'url': 'https://github.com/grahamjenson/list_of_recommender_systems', 'relevance_score': 0.77792513}



{'id': 'e3877564-0dcb-5cc3-be05-6ed27c5556c1', 'repo': 'recommenders-team/recommenders', 'description': 'Best Practices on Recommendation Systems', 'topics': ['ai', 'artificial-intellige

In [44]:
response4 = rag_graph.invoke({"question" : "What are some top deep learning libraries that I have starred on github?"})
print(response4['response'])

Based on your starred repositories, here are some of the top deep learning libraries you have bookmarked:

1. [Keras](https://github.com/keras-team/keras) — A user-friendly deep learning library for building neural networks in Python. It’s widely used for rapid experimentation and supports multiple backends like TensorFlow. You starred this because of its ease of use and flexibility in deep learning projects.

2. [PyTorch](https://github.com/pytorch/pytorch) — A popular, flexible deep learning framework known for dynamic computation graphs and GPU acceleration. Your star indicates you value its versatility for deep neural network development.

3. [TensorFlow](https://github.com/tensorflow/tensorflow) — Google's open-source framework for machine learning and deep learning, known for production deployment and scalability. Your starred repo suggests you’re interested in its comprehensive ecosystem.

4. [Fastai](https://github.com/fastai/fastai) — A high-level library built on PyTorch to s

In [45]:
for doc in response4['context']:
    print(doc.metadata)
    print("\n\n")

{'id': '88532728-a0cd-5d5f-8e54-68a9ab4a57b3', 'repo': 'lukasmasuch/best-of-ml-python', 'description': '🏆 A ranked list of awesome machine learning Python libraries. Updated weekly.', 'topics': ['automl', 'chatgpt', 'data-analysis', 'data-science', 'data-visualization', 'data-visualizations', 'deep-learning', 'gpt', 'gpt-3', 'jax', 'keras', 'machine-learning', 'ml', 'nlp', 'python', 'pytorch', 'scikit-learn', 'tensorflow', 'transformer'], 'language': None, 'doc_source': 'readme', 'stars': 23252, 'url': 'https://github.com/lukasmasuch/best-of-ml-python', '_id': '2b3e231e667d4a679c2496015880b6a9', '_collection_name': 'ask_my_bookmark', 'relevance_score': 0.75268626}



{'id': '601a986f-14da-5723-9df0-c827a8fdb834', 'repo': 'SkalskiP/courses', 'description': 'This repository is a curated collection of links to various courses and resources about Artificial Intelligence (AI)', 'topics': ['computer-vision', 'deep-learning', 'deep-neural-networks', 'generative-model', 'machine-learning', 'ml

In [36]:
response4['response']

"From the repositories you starred, some top deep learning libraries include:\n\n- [fastai](https://github.com/fastai/fastai): A high-level library built on PyTorch for fast and flexible deep learning. It's well-known for its ease of use and training speed.\n- [Keras](https://github.com/keras-team/keras): A user-friendly, high-level API for building and training deep neural networks, now integrated with TensorFlow.\n- [PyTorch](https://github.com/pytorch/pytorch): A popular, flexible deep learning framework with dynamic computation graphs, widely used in research and production.\n- [tensorflow](https://github.com/tensorflow/tensorflow): Google's comprehensive open-source machine learning framework, supporting a range of deep learning models.\n- [Jax](https://github.com/google/jax): A library for high-performance numerical computing with automatic differentiation, often used for research in deep learning.\n\nThese repositories are highly starred and central to the deep learning ecosyste

#### trying a newer hybrid search approach to deal with awesome lists/repos that are just links of other popular repositories

In [46]:
# ── 1. Metadata-only documents (description + topics, no README) ──────────────
CURATED_KEYWORDS = {"curated", "awesome", "ranked list", "collection of links", "resources"}

def is_curated_list(meta: dict) -> bool:
    desc = (meta.get("description") or "").lower()
    return any(kw in desc for kw in CURATED_KEYWORDS)

metadata_docs = [
    Document(
        page_content=(
            f"{meta.get('description') or ''}\n"
            f"Topics: {' '.join(meta.get('topics', []))}"
        ),
        metadata={**meta, "id": id_, "doc_type": "metadata"}
    )
    for text, id_, meta in zip(docs, ids, metadata)
]

# full-content documents — same as before but tag them
full_docs = [
    Document(
        page_content=text,
        metadata={**meta, "id": id_, "doc_type": "full"}
    )
    for text, id_, meta in zip(docs, ids, metadata)
]

# ── 2. BM25 retriever over metadata-only docs ─────────────────────────────────
bm25_metadata_retriever = BM25Retriever.from_documents(metadata_docs)
bm25_metadata_retriever.k = 5   # tighter — metadata is short, fewer candidates needed

# ── 3. Dense retriever over metadata-only docs ───────────────────────────────
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

meta_client = QdrantClient(":memory:")
meta_client.create_collection(
    collection_name="ask_my_bookmark_meta",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)
meta_vector_store = QdrantVectorStore(
    client=meta_client,
    collection_name="ask_my_bookmark_meta",
    embedding=embeddings,
)
meta_vector_store.add_documents(metadata_docs)
dense_metadata_retriever = meta_vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 10, "fetch_k": 25, "lambda_mult": 0.7},
)

# ── 4. Dense retriever over full-content docs (MMR for diversity) ─────────────
full_client = QdrantClient(":memory:")
full_client.create_collection(
    collection_name="ask_my_bookmark_full",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)
full_vector_store = QdrantVectorStore(
    client=full_client,
    collection_name="ask_my_bookmark_full",
    embedding=embeddings,
)
full_vector_store.add_documents(full_docs)
dense_full_retriever = full_vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 10, "fetch_k": 25, "lambda_mult": 0.7},
)

# ── 5. Three-way ensemble: metadata tracks dominate (0.70), full-content is fallback (0.30)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_metadata_retriever, dense_metadata_retriever, dense_full_retriever],
    weights=[0.25, 0.45, 0.30],
)

In [47]:
### build RAG graph
def format_context(docs) -> str:
    chunks = []
    for doc in docs:
        meta = doc.metadata
        topics = meta.get('topics', [])
        if topics:
            topics_str = f"Topics: {','.join(topics)}"
        else:
            topics_str = "N/A"
        chunk = f"""---
Repo: {meta.get('repo', 'N/A')}
URL: {meta.get('url', 'N/A')}
Description: {meta.get('description', 'N/A')}
Topics: {topics_str}
Programming Language: {meta.get('language', 'N/A')}
Stars: {meta.get('stars', 'N/A')}

README excerpt:
{doc.page_content.strip()}
---"""
        chunks.append(chunk)
    return "\n\n".join(chunks)


# define the RAG Pipeline in LangChain
RAG_SYSTEM_PROMPT = """
You are AskMyBookmark, a personal research assistant with access to the user's GitHub starred repositories.

Your job is to help the user discover, recall, and explore repositories they have bookmarked on GitHub. You answer questions by reasoning over the retrieved repository context provided to you — not from your general knowledge of what exists on GitHub.

**Ground rules:**
- Only surface repositories that are directly starred by the user. A directly starred repository is identified by the `repo` field in the retrieved context (e.g., `owner/repo-name`).
- Do NOT treat tools, libraries, papers, or projects that are merely *mentioned or listed inside* a repository's README as starred repositories. If a README is a curated list, awesome list, or ranked collection (e.g., its description contains phrases like "ranked list", "curated list", "awesome", "collection of links", or "resources"), treat that entire repository as a single starred item — do not extract or present its listed contents as individual starred repos.
- If no retrieved repositories are directly relevant to the query, say so honestly and suggest the user try rephrasing or broadening their search.
- You may use your general knowledge to explain a topic or technology, but all repository recommendations must come exclusively from the directly starred repos in the retrieved context.

**When presenting results:**
- Always include the repository's full name (Repo) as a markdown link to its GitHub URL: [Repo](URL)
- Include a brief description of what the repo does (from the description and topics fields), written in your own words if the original description is terse or absent.
- If the repository is a curated/awesome list, clearly label it as such (e.g., "curated list") so the user understands it is a collection, not a standalone library.
- Explain in 1–2 sentences *why* this repository is relevant to the user's query — this is the most important part.
- Group or rank results by relevance if there are several.
- If useful, note the primary programming language, star count, or topics to help the user evaluate the match.

**Tone:** Conversational, concise, and helpful. Treat the user as a developer who starred these repos intentionally and wants quick, intelligent recall — not a tutorial.

---

Retrieved repository context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(RAG_SYSTEM_PROMPT),
    HumanMessagePromptTemplate.from_template("{question}"),
])
llm = ChatOpenAI(model="gpt-4.1-nano")

def generate(state):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
    response = llm.invoke(messages)
    return {"response" : response.content}

class State(TypedDict):
    question: str
    context: List[Document]
    response: str

def retrieve(state):
    compressor = CohereRerank(model="rerank-v3.5")
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=ensemble_retriever, search_kwargs={"k": 10}
    )
    retrieved_docs = compression_retriever.invoke(state["question"])
    return {"context" : retrieved_docs}

# build graph
rag_graph_builder = StateGraph(State).add_sequence([retrieve, generate])
rag_graph_builder.add_edge(START, "retrieve")
rag_graph = rag_graph_builder.compile()

In [48]:
# evaluate responses
response = rag_graph.invoke({"question" : "Do I have any starred repositories related to Bayesian Statistics or Bayesian Modeling?"})
print(response['response'])

Yes, you do have starred repositories related to Bayesian Statistics and Bayesian Modeling. The most relevant one is:

- [PyMC Resources](https://github.com/pymc-devs/resources) — This repository provides educational resources for PyMC, including ports of various Bayesian modeling books like "Bayesian Data Analysis" by Gelman et al., "Statistical Rethinking" by Richard McElreath, and others. It's a valuable resource if you're interested in Bayesian modeling concepts and implementations in Python.

Additionally, the Orbit library, while mainly focused on Bayesian time series forecasting, involves Bayesian methods and probabilistic programming:

- [Orbit: A Python Package for Bayesian Forecasting](https://github.com/uber/orbit) — This is a stable, incubating project for Bayesian time series forecasting and inference, utilizing probabilistic programming languages under the hood.

Would you like more details on these repositories or suggestions for other related resources?


In [49]:
for doc in response['context']:
    print(doc.metadata)
    print("\n\n")

{'id': 'e901860a-7ff8-5671-84cc-2b760d76680b', 'repo': 'pymc-devs/pymc-resources', 'description': 'PyMC educational resources', 'topics': ['bayesian-inference', 'bayesian-statistics', 'data-analysis', 'data-science'], 'language': 'Jupyter Notebook', 'doc_source': 'readme', 'stars': 2076, 'url': 'https://github.com/pymc-devs/pymc-resources', 'doc_type': 'full', '_id': 'ce3f419d26bd42e5a163d69123affa63', '_collection_name': 'ask_my_bookmark_full', 'relevance_score': 0.49245724}



{'id': 'e721a9bf-8fee-5b98-9cbe-d4a38c1c3c91', 'repo': 'kidaufo/StatisticalModeling', 'description': 'Statistical modeling and Bayesian modeling by PyMC3, Stan and TensorFlow Probability', 'topics': [], 'language': 'Jupyter Notebook', 'doc_source': 'readme', 'stars': 23, 'url': 'https://github.com/kidaufo/StatisticalModeling', 'doc_type': 'metadata', 'relevance_score': 0.32110915}



{'id': 'f0432c99-3ba3-5970-afce-818508ce6074', 'repo': 'uber/orbit', 'description': 'A Python package for Bayesian forecasting wi

In [50]:
response2 = rag_graph.invoke({"question" : "What are some repos related to recommender systems that I have starred?"})
print(response2['response'])

Here are some repositories related to recommender systems that you've starred:

1. [listofRecommenderSystems](https://github.com/grahamjenson/listofrecommendersystems)  
   - A comprehensive list of open-source and academic recommender system tools, benchmarks, and applications.  
   - This repo is a curated collection that can help you explore various recommender system projects and methodologies.

2. [Deep-Learning-for-Recommendation-Systems](https://github.com/robi56/Deep-Learning-for-Recommendation-Systems)  
   - A collection of papers, tutorials, and code related to deep learning approaches for recommendation systems.  
   - Useful for exploring state-of-the-art deep learning techniques in recommender systems.

3. [RecommenderSystem-Paper](https://github.com/daicoolb/RecommenderSystem-Paper)  
   - A repository containing research papers, tools, and frameworks for recommender systems.  
   - Great for academic research and understanding different algorithms and techniques.

4. [R

In [51]:
for doc in response2['context']:
    print(doc.metadata)
    print("\n\n")

{'id': 'bc96cfad-0378-51f9-824a-1caadcd9c0f2', 'repo': 'jihoo-kim/awesome-RecSys', 'description': 'A curated list of awesome Recommender System (Books, Conferences, Researchers, Papers, Github Repositories, Useful Sites, Youtube Videos)', 'topics': [], 'language': None, 'doc_source': 'readme', 'stars': 1445, 'url': 'https://github.com/jihoo-kim/awesome-RecSys', 'doc_type': 'full', '_id': 'fbf48950c798471d81e7652c3c16223e', '_collection_name': 'ask_my_bookmark_full', 'relevance_score': 0.8302782}



{'id': 'e3877564-0dcb-5cc3-be05-6ed27c5556c1', 'repo': 'recommenders-team/recommenders', 'description': 'Best Practices on Recommendation Systems', 'topics': ['ai', 'artificial-intelligence', 'data-science', 'deep-learning', 'jupyter-notebook', 'kubernetes', 'machine-learning', 'operationalization', 'python', 'ranking', 'rating', 'recommendation', 'recommendation-algorithm', 'recommendation-engine', 'recommendation-system', 'recommender', 'tutorial'], 'language': 'Python', 'doc_source': 'rea

In [52]:
response3 = rag_graph.invoke({"question" : "What are the top recommender systems libraries that I have starred in the past?"})
print(response3['response'])

Based on your starred repositories, here are some of the top recommender system libraries you have bookmarked:

1. **Surprise**  
   [Surprise](https://github.com/NicolasHug/Surprise) — A Python library for building, analyzing, and experimenting with collaborative filtering recommender systems. It includes many algorithms like SVD, KNN, and more, making it a versatile tool for research and development in recommender systems.

2. **LightFM**  
   [LightFM](https://github.com/lyst/lightfm) — An actively-maintained Python library that implements hybrid recommendation algorithms combining collaborative and content-based approaches with efficient training, suitable for large datasets.

3. **Spotlight**  
   [Spotlight](https://github.com/maciejkula/spotlight) — A Python framework focused on matrix factorization and neural network-based models for recommendation systems, designed to be modular and extendable.

4. **recommenders**  
   [recommenders](https://github.com/recommenders-team/recom

In [53]:
for doc in response3['context']:
    print(doc.metadata)
    print("\n\n")

{'id': 'bc96cfad-0378-51f9-824a-1caadcd9c0f2', 'repo': 'jihoo-kim/awesome-RecSys', 'description': 'A curated list of awesome Recommender System (Books, Conferences, Researchers, Papers, Github Repositories, Useful Sites, Youtube Videos)', 'topics': [], 'language': None, 'doc_source': 'readme', 'stars': 1445, 'url': 'https://github.com/jihoo-kim/awesome-RecSys', 'doc_type': 'full', '_id': 'fbf48950c798471d81e7652c3c16223e', '_collection_name': 'ask_my_bookmark_full', 'relevance_score': 0.798077}



{'id': 'a77fae33-5ac7-5d81-81ce-597eec09e2d8', 'repo': 'grahamjenson/list_of_recommender_systems', 'description': 'A List of Recommender Systems and Resources', 'topics': [], 'language': None, 'doc_source': 'readme', 'stars': 4815, 'url': 'https://github.com/grahamjenson/list_of_recommender_systems', 'doc_type': 'full', '_id': 'c255ce3f78374c4690e8ea729938489a', '_collection_name': 'ask_my_bookmark_full', 'relevance_score': 0.77792513}



{'id': 'e3877564-0dcb-5cc3-be05-6ed27c5556c1', 'repo':

#### utilize a curated list to penalize github repos that are obviously just lists of other repos


In [62]:
CURATED_KEYWORDS = {"curated", "awesome", "ranked list", "collection of links", "resources", "list of", "best practices", "repositories for"}
MAX_CURATED_IN_RERANK = 0  # allow at most 2 curated list repos into the reranker


### build RAG graph
def format_context(docs) -> str:
    chunks = []
    for doc in docs:
        meta = doc.metadata
        topics = meta.get('topics', [])
        if topics:
            topics_str = f"Topics: {','.join(topics)}"
        else:
            topics_str = "N/A"
        chunk = f"""---
Repo: {meta.get('repo', 'N/A')}
URL: {meta.get('url', 'N/A')}
Description: {meta.get('description', 'N/A')}
Topics: {topics_str}
Programming Language: {meta.get('language', 'N/A')}
Stars: {meta.get('stars', 'N/A')}

README excerpt:
{doc.page_content.strip()}
---"""
        chunks.append(chunk)
    return "\n\n".join(chunks)


# define the RAG Pipeline in LangChain
RAG_SYSTEM_PROMPT = """
You are AskMyBookmark, a personal research assistant with access to the user's GitHub starred repositories.

Your job is to help the user discover, recall, and explore repositories they have bookmarked on GitHub. You answer questions by reasoning over the retrieved repository context provided to you — not from your general knowledge of what exists on GitHub.

**Ground rules:**
- Only surface repositories that are directly starred by the user. A directly starred repository is identified by the `repo` field in the retrieved context (e.g., `owner/repo-name`).
- Do NOT treat tools, libraries, papers, or projects that are merely *mentioned or listed inside* a repository's README as starred repositories. If a README is a curated list, awesome list, or ranked collection (e.g., its description contains phrases like "ranked list", "curated list", "awesome", "collection of links", or "resources"), treat that entire repository as a single starred item — do not extract or present its listed contents as individual starred repos.
- If no retrieved repositories are directly relevant to the query, say so honestly and suggest the user try rephrasing or broadening their search.
- You may use your general knowledge to explain a topic or technology, but all repository recommendations must come exclusively from the directly starred repos in the retrieved context.

**When presenting results:**
- Always include the repository's full name (Repo) as a markdown link to its GitHub URL: [Repo](URL)
- Include a brief description of what the repo does (from the description and topics fields), written in your own words if the original description is terse or absent.
- If the repository is a curated/awesome list, clearly label it as such (e.g., "curated list") so the user understands it is a collection, not a standalone library.
- Explain in 1–2 sentences *why* this repository is relevant to the user's query — this is the most important part.
- Group or rank results by relevance if there are several.
- If useful, note the primary programming language, star count, or topics to help the user evaluate the match.

**Tone:** Conversational, concise, and helpful. Treat the user as a developer who starred these repos intentionally and wants quick, intelligent recall — not a tutorial.

---

Retrieved repository context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(RAG_SYSTEM_PROMPT),
    HumanMessagePromptTemplate.from_template("{question}"),
])
llm = ChatOpenAI(model="gpt-4.1-nano")

def generate(state):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
    response = llm.invoke(messages)
    return {"response" : response.content}

class State(TypedDict):
    question: str
    context: List[Document]
    response: str


def is_curated_list(meta: dict) -> bool:
    desc = (meta.get("description") or "").lower()
    return any(kw in desc for kw in CURATED_KEYWORDS)

def retrieve(state):
    # Step 1: get all candidates from the ensemble
    candidates = ensemble_retriever.invoke(state["question"])

    # Step 2: split into focused repos and curated list repos
    focused = [doc for doc in candidates if not is_curated_list(doc.metadata)]
    curated = [doc for doc in candidates if is_curated_list(doc.metadata)]

    # Step 3: hard-cap curated repos — focused repos always dominate the rerank pool
    rerank_candidates = focused + curated[:MAX_CURATED_IN_RERANK]

    # Step 4: rerank the filtered candidate pool with Cohere
    compressor = CohereRerank(model="rerank-v3.5")
    reranked_docs = compressor.compress_documents(
        documents=rerank_candidates,
        query=state["question"],
    )

    return {"context": reranked_docs}


# build graph
rag_graph_builder = StateGraph(State).add_sequence([retrieve, generate])
rag_graph_builder.add_edge(START, "retrieve")
rag_graph = rag_graph_builder.compile()

In [63]:
# evaluate responses
response = rag_graph.invoke({"question" : "Do I have any starred repositories related to Bayesian Statistics or Bayesian Modeling?"})
print(response['response'])

Yes, you do have starred repositories related to Bayesian Statistics and Bayesian Modeling. The most relevant one is **Orbit**, which is a Python package specifically designed for Bayesian time series forecasting and inference. It supports various models like exponential smoothing, local-global trend, and uses probabilistic programming languages such as PyStan and Pyro under the hood. This repo is highly aligned with Bayesian methods and modeling.

Here's a quick overview:
- **[Orbit](https://github.com/uber/orbit)** — A Python library for Bayesian forecasting using probabilistic programming, supporting methods like MCMC, MAP, and VI. It focuses on time series and Bayesian inference.

Additionally, your "StatisticalModeling" repo might be relevant if it contains scripts or projects related to Bayesian statistics, but from the description, it seems more general-purpose or personal. The standout starred repository in Bayesian modeling is Orbit.


In [64]:
for doc in response['context']:
    print(doc.metadata)
    print("\n\n")

{'id': 'e721a9bf-8fee-5b98-9cbe-d4a38c1c3c91', 'repo': 'kidaufo/StatisticalModeling', 'description': 'Statistical modeling and Bayesian modeling by PyMC3, Stan and TensorFlow Probability', 'topics': [], 'language': 'Jupyter Notebook', 'doc_source': 'readme', 'stars': 23, 'url': 'https://github.com/kidaufo/StatisticalModeling', 'doc_type': 'metadata', 'relevance_score': 0.32110915}



{'id': 'f0432c99-3ba3-5970-afce-818508ce6074', 'repo': 'uber/orbit', 'description': 'A Python package for Bayesian forecasting with object-oriented design and probabilistic models under the hood.', 'topics': ['arima', 'bayesian', 'bayesian-methods', 'bayesian-statistics', 'changepoint', 'exponential-smoothing', 'forecast', 'forecasting', 'machine-learning', 'orbit', 'probabilistic', 'probabilistic-programming', 'pyro', 'pystan', 'python', 'pytorch', 'regression', 'regression-models', 'stan', 'time-series'], 'language': 'Python', 'doc_source': 'readme', 'stars': 2038, 'url': 'https://github.com/uber/orbit',

In [65]:
response2 = rag_graph.invoke({"question" : "What are some repos related to recommender systems that I have starred?"})
print(response2['response'])

You have starred a repository dedicated to public datasets for recommender systems. It's a comprehensive collection of high-quality, topic-centric datasets from various sources like Amazon, MovieLens, Yahoo Music, and more. The repo also includes pre-processed datasets suitable for academic experiments, making it a valuable resource for anyone working on recommender systems. 

**Repository: [Public Datasets For Recommender Systems](https://github.com/your-user/repo-name)**  
- **Description:** A curated list of high-quality datasets from multiple domains (books, movies, music, e-commerce, etc.) for recommender systems, collected from academic sources and online platforms.  
- **Relevance:** Since recommender systems rely heavily on quality datasets, this repo is a crucial resource for research, development, and experimentation in the field of recommendation algorithms.


In [67]:
for doc in response2['context']:
    print(doc.metadata)
    print("\n\n")

{'id': 'ebc556a7-f741-5dd8-bf4f-cddd2cdc22e4', 'repo': 'RUCAIBox/RecSysDatasets', 'description': 'This is a repository of public data sources for Recommender Systems (RS).', 'topics': ['atomic-files', 'dataset', 'recbole', 'recommendation-datasets', 'recommendations', 'recommender-system'], 'language': 'Python', 'doc_source': 'readme', 'stars': 1167, 'url': 'https://github.com/RUCAIBox/RecSysDatasets', 'doc_type': 'metadata', '_id': '13d84d46781341649fdf88a39e2c18d2', '_collection_name': 'ask_my_bookmark_meta', 'relevance_score': 0.6991933}



{'id': 'b8489869-3116-518b-b412-c389c030711f', 'repo': 'caserec/Datasets-for-Recommender-Systems', 'description': 'This is a repository of a topic-centric public data sources in high quality for Recommender Systems (RS)', 'topics': ['data-science', 'database', 'datasets', 'public-data', 'recommender-systems'], 'language': 'Jupyter Notebook', 'doc_source': 'readme', 'stars': 1091, 'url': 'https://github.com/caserec/Datasets-for-Recommender-Systems

In [68]:
response3 = rag_graph.invoke({"question" : "What are the top recommender systems libraries that I have starred in the past?"})
print(response3['response'])

Based on your starred repositories, the top recommender systems libraries you have starred include:

1. [LibRecommender](https://github.com/massquantity/LibRecommender) — This is a comprehensive, easy-to-use recommender system library that supports various algorithms like FM, DIN, LightGCN, and hybrid models. It’s built for end-to-end recommendation workflows, including data handling, model training, and deployment, with a focus on collaborative filtering and content-based features.
   
2. [TensorFlow Recommenders](https://github.com/tensorflow/recommenders) — A well-regarded library designed for building recommender system models using TensorFlow. It simplifies the entire workflow from data preparation to deployment and is built on Keras, making it flexible for complex model development.

These repositories are prominent tools for developing advanced recommendation systems, and your star indicates a strong interest in the field. Would you like me to explore other related libraries or 

In [69]:
for doc in response3['context']:
    print(doc.metadata)
    print("\n\n")

{'id': 'b7a4ad3f-bffa-5e3d-a38c-8e6572e8427a', 'repo': 'massquantity/LibRecommender', 'description': 'Versatile End-to-End Recommender System', 'topics': ['collaborative-filtering', 'cython', 'recommender-system', 'rust'], 'language': 'Python', 'doc_source': 'readme', 'stars': 467, 'url': 'https://github.com/massquantity/LibRecommender', 'doc_type': 'full', '_id': '3607d210d42c4fc1895a6c6a3e14918f', '_collection_name': 'ask_my_bookmark_full', 'relevance_score': 0.5063723}



{'id': 'e71b4593-e25d-53c5-a1e0-dc2492600dd6', 'repo': 'RUCAIBox/RecBole', 'description': 'A unified, comprehensive and efficient recommendation library', 'topics': ['collaborative-filtering', 'ctr-prediction', 'deep-learning', 'graph-neural-networks', 'knowledge-graph', 'pytorch', 'recommendation-system', 'recommendations', 'recommender', 'recommender-systems', 'sequential-recommendation'], 'language': 'Python', 'doc_source': 'readme', 'stars': 4286, 'url': 'https://github.com/RUCAIBox/RecBole', 'doc_type': 'metad

#### Push filtering of curated list repos to the vectorDB instead (store on index instead of applying during query time)


In [78]:
# ── Curated list classification ───────────────────────────────────────────────
CURATED_KEYWORDS = {"curated", "awesome", "ranked list", "collection of links", "resources", "list of", "best practices", "repositories for", "papers", "articles", "repositories"}
MAX_CURATED_IN_RERANK = 0  # allow at most 2 curated list repos into the reranker

def classify_repo(meta: dict) -> str:
    desc = (meta.get("description") or "").lower()
    return "curated_list" if any(kw in desc for kw in CURATED_KEYWORDS) else "library"

def is_curated_list(meta: dict) -> bool:
    return classify_repo(meta) == "curated_list"

# ── 1. Build metadata-only documents (description + topics, no README) ────────
metadata_docs = [
    Document(
        page_content=(
            f"{meta.get('description') or ''}\n"
            f"Topics: {' '.join(meta.get('topics', []))}"
        ),
        metadata={**meta, "id": id_, "repo_type": classify_repo(meta)}
    )
    for text, id_, meta in zip(docs, ids, metadata)
]

# ── 2. Build full-content documents (description + README, same as before) ────
full_docs = [
    Document(
        page_content=text,
        metadata={**meta, "id": id_, "repo_type": classify_repo(meta)}
    )
    for text, id_, meta in zip(docs, ids, metadata)
]

# ── 3. BM25 over metadata-only docs ──────────────────────────────────────────
bm25_metadata_retriever = BM25Retriever.from_documents(metadata_docs)
bm25_metadata_retriever.k = 5

# ── 4. Dense retriever over metadata-only docs (Qdrant pre-filters curated repos) ──
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

meta_client = QdrantClient(":memory:")
meta_client.create_collection(
    collection_name="ask_my_bookmark_meta",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)
meta_vector_store = QdrantVectorStore(
    client=meta_client,
    collection_name="ask_my_bookmark_meta",
    embedding=embeddings,
)
meta_vector_store.add_documents(metadata_docs)

from qdrant_client.http.models import Filter, FieldCondition, MatchValue

dense_metadata_retriever = meta_vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 10,
        "fetch_k": 25,
        "lambda_mult": 0.7,
        "filter": Filter(
            must_not=[
                FieldCondition(
                    key="metadata.repo_type",
                    match=MatchValue(value="curated_list")
                )
            ]
        ),
    },
)

# ── 5. Dense retriever over full-content docs (MMR, curated repos allowed for recall) ──
full_client = QdrantClient(":memory:")
full_client.create_collection(
    collection_name="ask_my_bookmark_full",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)
full_vector_store = QdrantVectorStore(
    client=full_client,
    collection_name="ask_my_bookmark_full",
    embedding=embeddings,
)
full_vector_store.add_documents(full_docs)

dense_full_retriever = full_vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 10, "fetch_k": 25, "lambda_mult": 0.7},
)

# ── 6. Three-way ensemble via RRF ─────────────────────────────────────────────
# metadata tracks dominate (0.70 combined); full-content is recall fallback (0.30)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_metadata_retriever, dense_metadata_retriever, dense_full_retriever],
    weights=[0.25, 0.45, 0.30],
)


### build RAG graph
def format_context(docs) -> str:
    chunks = []
    for doc in docs:
        meta = doc.metadata
        topics = meta.get('topics', [])
        if topics:
            topics_str = f"Topics: {','.join(topics)}"
        else:
            topics_str = "N/A"
        chunk = f"""---
Repo: {meta.get('repo', 'N/A')}
URL: {meta.get('url', 'N/A')}
Description: {meta.get('description', 'N/A')}
Topics: {topics_str}
Programming Language: {meta.get('language', 'N/A')}
Stars: {meta.get('stars', 'N/A')}

README excerpt:
{doc.page_content.strip()}
---"""
        chunks.append(chunk)
    return "\n\n".join(chunks)


# define the RAG Pipeline in LangChain
RAG_SYSTEM_PROMPT = """
You are AskMyBookmark, a personal research assistant with access to the user's GitHub starred repositories.

Your job is to help the user discover, recall, and explore repositories they have bookmarked on GitHub. You answer questions by reasoning over the retrieved repository context provided to you — not from your general knowledge of what exists on GitHub.

**Ground rules:**
- Only surface repositories that are directly starred by the user. A directly starred repository is identified by the `repo` field in the retrieved context (e.g., `owner/repo-name`).
- Do NOT treat tools, libraries, papers, or projects that are merely *mentioned or listed inside* a repository's README as starred repositories. If a README is a curated list, awesome list, or ranked collection (e.g., its description contains phrases like "ranked list", "curated list", "awesome", "collection of links", or "resources"), treat that entire repository as a single starred item — do not extract or present its listed contents as individual starred repos.
- If no retrieved repositories are directly relevant to the query, say so honestly and suggest the user try rephrasing or broadening their search.
- You may use your general knowledge to explain a topic or technology, but all repository recommendations must come exclusively from the directly starred repos in the retrieved context.

**When presenting results:**
- Always include the repository's full name (Repo) as a markdown link to its GitHub URL: [Repo](URL)
- Include a brief description of what the repo does (from the description and topics fields), written in your own words if the original description is terse or absent.
- If the repository is a curated/awesome list, clearly label it as such (e.g., "curated list") so the user understands it is a collection, not a standalone library.
- Explain in 1–2 sentences *why* this repository is relevant to the user's query — this is the most important part.
- Group or rank results by relevance if there are several.
- If useful, note the primary programming language, star count, or topics to help the user evaluate the match.

**Tone:** Conversational, concise, and helpful. Treat the user as a developer who starred these repos intentionally and wants quick, intelligent recall — not a tutorial.

---

Retrieved repository context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(RAG_SYSTEM_PROMPT),
    HumanMessagePromptTemplate.from_template("{question}"),
])
llm = ChatOpenAI(model="gpt-4.1-nano")

def generate(state):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
    response = llm.invoke(messages)
    return {"response" : response.content}

class State(TypedDict):
    question: str
    context: List[Document]
    response: str


# ── 7. Retrieve function with pre-Cohere curated repo cap ─────────────────────
def retrieve(state):
    candidates = ensemble_retriever.invoke(state["question"])

    focused = [doc for doc in candidates if not is_curated_list(doc.metadata)]
    curated = [doc for doc in candidates if is_curated_list(doc.metadata)]

    # focused repos fill the rerank pool; curated repos capped at MAX_CURATED_IN_RERANK
    rerank_candidates = focused + curated[:MAX_CURATED_IN_RERANK]

    compressor = CohereRerank(model="rerank-v3.5")
    reranked_docs = compressor.compress_documents(
        documents=rerank_candidates,
        query=state["question"],
    )
    return {"context": reranked_docs}


# build graph
rag_graph_builder = StateGraph(State).add_sequence([retrieve, generate])
rag_graph_builder.add_edge(START, "retrieve")
rag_graph = rag_graph_builder.compile()

In [79]:
# evaluate responses
response = rag_graph.invoke({"question" : "Do I have any starred repositories related to Bayesian Statistics or Bayesian Modeling?"})
print(response['response'])

Yes, you do have starred repositories related to Bayesian Statistics and Bayesian Modeling. 

The most relevant one is [orbit](https://github.com/uber/orbit), which is a Python package for Bayesian time series forecasting and inference. It provides tools for Bayesian modeling using probabilistic programming languages, supporting models like exponential smoothing and trend models, with inference methods such as MCMC and Variational Inference. Its topics include Bayesian methods, Bayesian statistics, and probabilistic programming, making it highly relevant to your interests.


In [80]:
for doc in response['context']:
    print(doc.metadata)
    print("\n\n")

{'id': 'e721a9bf-8fee-5b98-9cbe-d4a38c1c3c91', 'repo': 'kidaufo/StatisticalModeling', 'description': 'Statistical modeling and Bayesian modeling by PyMC3, Stan and TensorFlow Probability', 'topics': [], 'language': 'Jupyter Notebook', 'doc_source': 'readme', 'stars': 23, 'url': 'https://github.com/kidaufo/StatisticalModeling', 'repo_type': 'library', 'relevance_score': 0.32110915}



{'id': 'f0432c99-3ba3-5970-afce-818508ce6074', 'repo': 'uber/orbit', 'description': 'A Python package for Bayesian forecasting with object-oriented design and probabilistic models under the hood.', 'topics': ['arima', 'bayesian', 'bayesian-methods', 'bayesian-statistics', 'changepoint', 'exponential-smoothing', 'forecast', 'forecasting', 'machine-learning', 'orbit', 'probabilistic', 'probabilistic-programming', 'pyro', 'pystan', 'python', 'pytorch', 'regression', 'regression-models', 'stan', 'time-series'], 'language': 'Python', 'doc_source': 'readme', 'stars': 2038, 'url': 'https://github.com/uber/orbit',

In [81]:
response2 = rag_graph.invoke({"question" : "What are some repos related to recommender systems that I have starred?"})
print(response2['response'])

Based on your starred repositories, you have a few related to recommender systems:

1. [Public Data Sources for Recommender Systems](https://github.com/arthurfortes/datasets-recommender-systems)  
   This repository collects high-quality, public datasets specifically for recommender system research. It includes datasets from various domains like books, dating, e-commerce, music, movies, games, jokes, food, and more. It’s valuable if you're looking for datasets to experiment with or analyze in recommender system projects.

This appears to be the main repo directly related to recommender systems that you've starred. If you have other starred repos that are more specific or different, let me know!


In [83]:
for doc in response2['context']:
    print(doc.metadata)
    print("\n\n")

{'id': 'ebc556a7-f741-5dd8-bf4f-cddd2cdc22e4', 'repo': 'RUCAIBox/RecSysDatasets', 'description': 'This is a repository of public data sources for Recommender Systems (RS).', 'topics': ['atomic-files', 'dataset', 'recbole', 'recommendation-datasets', 'recommendations', 'recommender-system'], 'language': 'Python', 'doc_source': 'readme', 'stars': 1167, 'url': 'https://github.com/RUCAIBox/RecSysDatasets', 'repo_type': 'library', '_id': '717de2be640c4d109e1cd2bc87f04e4f', '_collection_name': 'ask_my_bookmark_meta', 'relevance_score': 0.6991933}



{'id': 'b8489869-3116-518b-b412-c389c030711f', 'repo': 'caserec/Datasets-for-Recommender-Systems', 'description': 'This is a repository of a topic-centric public data sources in high quality for Recommender Systems (RS)', 'topics': ['data-science', 'database', 'datasets', 'public-data', 'recommender-systems'], 'language': 'Jupyter Notebook', 'doc_source': 'readme', 'stars': 1091, 'url': 'https://github.com/caserec/Datasets-for-Recommender-Systems

In [84]:
response3 = rag_graph.invoke({"question" : "What are the top recommender systems libraries that I have starred in the past?"})
print(response3['response'])

Based on the repositories you've starred, here are some top recommender system libraries:

1. [LibRecommender](https://github.com/massquantity/LibRecommender) — A comprehensive, easy-to-use recommender system library supporting various algorithms like FM, DIN, LightGCN, and hybrid approaches. It covers end-to-end processes including data handling, training, and deployment, with features for both collaborative filtering and content-based models. This appears to be the primary, all-encompassing library you've starred for building recommender systems.

2. [TensorFlow Recommenders](https://github.com/tensorflow/recommenders) — A flexible library built on TensorFlow and Keras for developing recommender models. It supports the full pipeline from data preparation to deployment, offering a suite of tools and prebuilt models like matrix factorization and retrieval-based methods.

These libraries are among the most prominent and feature-rich options in your starred list for building and experime

In [86]:
for doc in response3['context']:
    print(doc.metadata)
    print("\n\n")

{'id': 'b7a4ad3f-bffa-5e3d-a38c-8e6572e8427a', 'repo': 'massquantity/LibRecommender', 'description': 'Versatile End-to-End Recommender System', 'topics': ['collaborative-filtering', 'cython', 'recommender-system', 'rust'], 'language': 'Python', 'doc_source': 'readme', 'stars': 467, 'url': 'https://github.com/massquantity/LibRecommender', 'repo_type': 'library', '_id': '6bdd7ac5cdbd4135a53b1582c191b0e9', '_collection_name': 'ask_my_bookmark_full', 'relevance_score': 0.5063723}



{'id': 'e71b4593-e25d-53c5-a1e0-dc2492600dd6', 'repo': 'RUCAIBox/RecBole', 'description': 'A unified, comprehensive and efficient recommendation library', 'topics': ['collaborative-filtering', 'ctr-prediction', 'deep-learning', 'graph-neural-networks', 'knowledge-graph', 'pytorch', 'recommendation-system', 'recommendations', 'recommender', 'recommender-systems', 'sequential-recommendation'], 'language': 'Python', 'doc_source': 'readme', 'stars': 4286, 'url': 'https://github.com/RUCAIBox/RecBole', 'repo_type': '